# ПР-11. Понижение размерности методом PCA

Сжать коррелирующие признаки, изучить долю объяснённой дисперсии и сравнить качество классификации до и после PCA.

**Данные:** см. README проекта. Выполняйте ячейки сверху вниз; фиксируйте результаты в отчёте.

## 1. Подготовка данных

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent
data = pd.read_csv(ROOT / 'data' / 'classification_breast_cancer.csv')
X = data.drop(columns=['target', 'target_name'])
y = data['target'].astype(int)  # 0 = malignant, 1 = benign
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print('train/test:', X_train.shape, X_test.shape)


## 1. Доля объяснённой дисперсии

PCA чувствителен к масштабу, поэтому сначала стандартизируем признаки. Подбор PCA производится только на train.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
scaler=StandardScaler().fit(X_train)
Ztr=scaler.transform(X_train); Zte=scaler.transform(X_test)
pca=PCA().fit(Ztr)
cum=pca.explained_variance_ratio_.cumsum()
plt.plot(range(1,len(cum)+1),cum,marker='.')
plt.axhline(.90,color='gray',ls='--'); plt.axhline(.95,color='gray',ls=':')
plt.xlabel('Число компонент'); plt.ylabel('Накопленная доля дисперсии'); plt.grid(True); plt.show()
print('Компонент для 90%:',int(np.searchsorted(cum,.90)+1))

## 2. Качество downstream-классификации

Сравните LogisticRegression с исходными масштабированными признаками и PCA с несколькими числами компонент. Все операции fit выполняются на train.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import cross_val_score,StratifiedKFold
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
for n in [5,10,15,20,30]:
    model=Pipeline([('scale',StandardScaler()),('pca',PCA(n_components=n)),
                    ('clf',LogisticRegression(max_iter=3000,random_state=42))])
    cvf=cross_val_score(model,X_train,y_train,cv=cv,scoring='f1_macro').mean()
    model.fit(X_train,y_train); testf=f1_score(y_test,model.predict(X_test),average='macro')
    print(n,'CV F1 macro=',round(cvf,3),'test F1 macro=',round(testf,3))

## Контрольные вопросы и вывод

1. Какое число компонент сохраняет 90% дисперсии?
2. Сохранилось ли качество классификации при сокращении размерности?
3. Почему PCA не использует y при поиске компонент?
4. Назовите компромисс между сжатием, информацией и устойчивостью.

**В отчёте:** график накопленной дисперсии, таблица числа компонент/CV/test F1 и обоснованный компромисс.